# Load Data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

''' ---------------------------------------------------------------------------
    File description
'''

data = np.load(Path("..") / "data" / "mechanical_fingerprint.npz")

params = ["A", "B", "alpha", "beta", "E", "gamma"]
dirs = ["x", "y"]

log_params = {"A", "B", "E"}

dfs = {}

symmetry_x = 162.5

''' ---------------------------------------------------------------------------
    Symmetry and applying logarithm
'''

for d in dirs:
    coords_key = "coords_x_trans_ds_2" if d == "x" else "coords_y_trans_ds_2"
    coords = data[coords_key]

    df_d = pd.DataFrame({
        "cell_idx": data["cell_idx"],
        "cell_label": data["cell_label_in_day_folder"],
        "x_pos": coords[:, 0] - symmetry_x,
        "y_pos": coords[:, 1],
    })

    for p in params:
        values = data[f"{p}_{d}"]

        if p in log_params:
            df_d[f"log{p}"] = np.log(values)
        else:
            df_d[p] = values

    dfs[d] = df_d

''' ---------------------------------------------------------------------------
    Cleanup
'''

df_x = dfs["x"].replace([np.inf, -np.inf], np.nan).dropna()
df_y = dfs["y"].replace([np.inf, -np.inf], np.nan).dropna()

# PCA

Same PCA as in `q1 x.ipynb` / `q1 y.ipynb`, with all principal components sorted by explained variance (PC1 first). Each component is oriented so that its beta loading is positive (for PC1: larger values = more fluid-like).

In [2]:
fields = ["logA", "logB", "alpha", "beta", "logE", "gamma"]
rows = ["logA", "logB", "logE", "alpha", "beta", "gamma"]

def pca(df):

    ''' z normalized fields '''

    Z = df[fields].to_numpy()
    Z = (Z - Z.mean(axis=0)) / Z.std(axis=0)

    ''' Principal components, PC1 first '''

    eigval, eigvec = np.linalg.eigh(np.cov(Z, rowvar=False))
    order = np.argsort(eigval)[::-1]
    eigval, eigvec = eigval[order], eigvec[:, order]

    eigvec = eigvec * np.where(eigvec[3] < 0, -1, 1)   # beta loading positive

    pcs = [f"PC{i + 1}" for i in range(len(eigval))]

    loadings = pd.DataFrame(eigvec, index=fields, columns=pcs).loc[rows]

    exp_var = pd.DataFrame({
        "explained variance (%)": eigval / eigval.sum() * 100,
    }, index=pcs)

    return loadings, exp_var

# x direction

In [3]:
loadings_x, exp_var_x = pca(df_x)
display(loadings_x.round(4))
display((loadings_x**2 * 100).add_suffix("² (%)").round(2))
display(exp_var_x.round(2))

,PC1,PC2,PC3,PC4,PC5,PC6
logA,-0.3743,0.3559,0.4576,0.6348,-0.3089,-0.1596
logB,-0.5163,-0.1560,-0.3536,-0.0424,-0.4741,0.5979
logE,-0.1744,0.7010,0.1697,-0.6632,-0.0970,0.0088
alpha,0.5160,0.0244,-0.1702,-0.0415,-0.7905,-0.2784
beta,0.5318,0.1788,0.3536,0.1478,0.0104,0.7337
gamma,0.1180,0.5701,-0.6948,0.3631,0.2130,0.0343


,PC1² (%),PC2² (%),PC3² (%),PC4² (%),PC5² (%),PC6² (%)
logA,14.01,12.67,20.94,40.29,9.54,2.55
logB,26.66,2.43,12.50,0.18,22.47,35.75
logE,3.04,49.14,2.88,43.99,0.94,0.01
alpha,26.63,0.06,2.90,0.17,62.49,7.75
beta,28.28,3.20,12.51,2.19,0.01,53.82
gamma,1.39,32.50,48.28,13.18,4.54,0.12


,explained variance (%)
PC1,47.65
PC2,22.37
PC3,15.25
PC4,8.48
PC5,5.53
PC6,0.71


# y direction

In [4]:
loadings_y, exp_var_y = pca(df_y)
display(loadings_y.round(4))
display((loadings_y**2 * 100).add_suffix("² (%)").round(2))
display(exp_var_y.round(2))

,PC1,PC2,PC3,PC4,PC5,PC6
logA,-0.3518,0.3193,0.6273,0.5524,-0.2131,-0.1736
logB,-0.5364,-0.1663,-0.1518,-0.0882,-0.5340,0.6072
logE,-0.1965,0.6587,0.1917,-0.6978,0.0610,0.0071
alpha,0.5077,0.0181,0.0874,-0.1744,-0.7998,-0.2534
beta,0.5387,0.2155,0.3005,0.1517,0.1144,0.7327
gamma,0.0437,0.6242,-0.6699,0.3829,-0.1140,-0.0026


,PC1² (%),PC2² (%),PC3² (%),PC4² (%),PC5² (%),PC6² (%)
logA,12.38,10.20,39.35,30.52,4.54,3.02
logB,28.77,2.77,2.31,0.78,28.52,36.87
logE,3.86,43.39,3.67,48.69,0.37,0.01
alpha,25.78,0.03,0.76,3.04,63.96,6.42
beta,29.02,4.65,9.03,2.30,1.31,53.69
gamma,0.19,38.96,44.88,14.66,1.30,0.00


,explained variance (%)
PC1,47.32
PC2,23.04
PC3,13.86
PC4,8.85
PC5,6.33
PC6,0.60
